In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.nn import functional as F

In [ ]:
batch_size = 64 #indep sequence processing in parallel
block_size = 256 #maximum context length for prediction
max_iters = 5000
eval_interval = 500
lr = 3e-4
device ='cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_emb_dim = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [ ]:
print(device)

In [ ]:
torch.manual_seed(3407)

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Length (char):", len(text))
print(text[:500])

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [ ]:
print(vocab_size)

In [ ]:
str_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_str = { i:ch for i, ch in enumerate(chars) }

In [ ]:
encode = lambda s: [str_to_int[c] for c in s] #take a str and output a list of int

In [ ]:
decode = lambda l: ''.join([int_to_str[i] for i in l]) #vice versa

In [ ]:
print(encode("Meoww"))

In [ ]:
print(decode([25, 43, 53, 61, 61]))

ENCODING ENTIRE TEXT DATASET and then store it into torch.Tensor

In [ ]:
data = torch.tensor(encode(text), dtype =torch.long)
print(data.shape, data.dtype)

In [ ]:
print(data[:100])

In [ ]:
# train, val = train_test_split(data, train_size=0.9, random_state=0)

n = int(0.9*len(data)) # first 90% will be train
train = data[:n]
val= data[n:]

In [ ]:
print(train.shape)
print(val.shape)

In [ ]:
# block_size = 8
# train[:block_size+1] # efficiency + for making TF see context from as little as 1 to block size

#tf will never see more than block_size input and op is truncated. context for any thing between 1 to blocksize can be inferenced

we will have mini batches of multiple chunks of text stacked up on a single tensor. FOR efficiency and parallelization. KEEP GPU BUSY.

In [ ]:
def get_batch (split):
    data = train if split == 'train' else val
    ix = torch.randint( len(data) - block_size, (batch_size,))
    x = torch.stack(
        [data[i: i+block_size] for i in ix]
    )
    y = torch.stack(
            [data[i+1: i+block_size+1] for i in ix]
    )
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
@torch.no_grad() #dont call backward which is mroe efficet, no need to store intermediate var
def estimate_loss():
    output = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[i] = loss.item()
        output[split] = losses.mean()
    model.train()
    return output

SELF ATTENTION HEAD

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb_dim, head_size, bias = False)
        self.query = nn.Linear(n_emb_dim, head_size, bias = False)
        self.value = nn.Linear(n_emb_dim, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) #lower trig matrix

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        #compute attention scires along with normalisation with dimension

        weights = q @ k.transpose( -2, -1) * C ** -0.5  #(B, T, C)  (B, C, T) -> (B, T, T)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) #for decoder block to mask out unwanted context
        weights = F.softmax(weights, dim = -1) #B, T, T

        #WEIGHTERD AGGREGATION
        v = self.value(x)
        output = weights @ v
        return output

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList( 
            [Head(head_size) for _ in range(num_heads)]
        )
        self.proj = nn.Linear(n_emb_dim, n_emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads], dim = -1
        ) #concat over channel dim
        out = self.proj(out) #linear tf
        return out

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, n_emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb_dim,4*n_emb_dim),
            nn.ReLU(),
            nn.Linear(4*n_emb_dim, n_emb_dim), #projection layer
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x) #per token level ie all the tokens do this independently (think on the data individually)

In [ ]:
class Block(nn.Module):

    def __init__(self, n_emb_dim, n_head):
        super().__init__()
        head_size = n_emb_dim // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_emb_dim)
        self.ln1 = nn.LayerNorm(n_emb_dim)
        self.ln2 = nn.LayerNorm(n_emb_dim) #32 size ; mean and var over 32 numbers

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb_dim)
        self.position_embedding_table = nn.Embedding(block_size, n_emb_dim)
        self.blocks = nn.Sequential(*
            [Block(n_emb_dim, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_emb_dim)
        self.lm_head = nn.Linear(n_emb_dim, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape
        #idx and targets are both (B, T) tensor
        token_emb = self.token_embedding_table(idx)  #(B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device)) # T, C ; INT from 0 to T-1; ALL these int get embeedded
        x = token_emb + pos_emb  #B,T,C

        #apply multi head self attention (B, T, ,C)
        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x) #(B, T, vocab_size) 

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):

            #crop idx upto the last block size
            crop_idx = idx[:, -block_size:]
            
            # get the predictions
            logits, loss = self(crop_idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [ ]:
model = BigramLanguageModel()
m = model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr = lr) #3e-4 for most networks but for small this works or even higher

In [ ]:
for iter in range(max_iters):

    #every once in a while eval the loss on train and val sets
    if iter %eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #sample a batch of data
    xb, yb = get_batch('train')

    #eval the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) #setting grads in previous step to zero
    loss.backward() #get grad for all the param
    optimizer.step() #use grad to update param

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=300)[0].tolist()))